# 标签数据

In [2]:
import pandas as pd
import os
import glob
import numpy as np
import h5py

In [17]:

def convert_csv_format(input_folder, output_file):
    """
    将多个CSV文件从原始格式转换为目标格式并合并
    
    参数:
        input_folder: 输入CSV文件所在的文件夹路径
        output_file: 输出CSV文件的路径
    """
    # 获取所有CSV文件
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    
    # 初始化一个空的数据框来存储所有数据
    all_data = pd.DataFrame()
    
    # 处理每个CSV文件
    for file_path in csv_files:
        try:
            # 读取CSV文件
            df = pd.read_csv(file_path, sep=',',header=None, names=['ch', 'phase_index', 'target', 'mag', 'phase'])
            S_df = df[df['phase']=='S'].copy()
            S_df.rename(columns={'phase_index':'s_target'},inplace = True)

            P_df = df[df['phase']=='P'].copy()
            P_df.rename(columns={'phase_index':'p_target'},inplace = True)
            merge_df = pd.merge(S_df, P_df, how='inner' , on=['ch'])
           

            
            # 创建目标格式的数据框
            converted_df = pd.DataFrame()
            
            # 转换格式
            # Key: 可能需要根据Key1和Key2组合而成，这里假设是Key1_Key2
            converted_df['Key'] = file_path.split("/")[-1][:-4] + '_' + merge_df['ch'].astype(str)
            
            # p_target和s_target: 从target和phase计算得到
            # 假设target是幅度，phase是相位，那么:
            # p_target = target * cos(phase)
            # s_target = target * sin(phase)
            converted_df['p_target'] = merge_df['p_target']
            converted_df['s_target'] = merge_df['s_target']
            
            # Dis: 可能需要计算或保留为空
            converted_df['Dis'] = np.nan  # 暂时设为NaN
            
            # Mag_value: 直接使用mag列
            converted_df['Mag_value'] = merge_df['mag_x']
            
            # From: 记录来源文件名
            converted_df['From'] = "xfj3km"
            
            # snr: 可能需要计算或保留为空
            converted_df['snr'] = np.nan  # 暂时设为NaN
            
            # datasplit: 可能需要根据某些规则分配
            converted_df['datasplit'] = 'train'  # 默认设为train
            
            # shot: 可能需要从文件名或其他信息提取
            converted_df['shot'] = 100  # 默认设为1
            
            # 添加到总数据框
            all_data = pd.concat([all_data, converted_df], ignore_index=True)
            # print(converted_df)
            print(f"已处理文件: {file_path}")
            
        except Exception as e:
            print(f"处理文件 {file_path} 时出错: {str(e)}")
    
    # 保存合并后的数据
    if not all_data.empty:
        
        all_data.loc[int(len(all_data)*0.9):, "datasplit"] = "val"
        filter_exist = True
        if filter_exist :
            h5_file_path = "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_6w.h5"
            dataset_names = []
            with h5py.File(h5_file_path, 'r') as h5f:
                dataset_names = list(h5f.keys())
            all_data = all_data[all_data['Key'].isin(dataset_names)]
        all_data.to_csv(output_file, index=False)
        print(f"已保存合并后的数据到: {output_file}")
        print(f"总记录数: {len(all_data)}")
    else:
        print("没有找到任何有效数据")

# 使用示例
if __name__ == "__main__":
    input_folder = "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks"  # 替换为您的CSV文件所在文件夹路径
    output_file = "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_6w.csv"  # 替换为您想要的输出文件路径
    
    convert_csv_format(input_folder, output_file)

已处理文件: /home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/xfj_das_100Hz_149.csv
已处理文件: /home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/xfj_das_100Hz_150.csv
已处理文件: /home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/xfj_das_100Hz_151.csv
已处理文件: /home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/xfj_das_100Hz_152.csv
已处理文件: /home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/xfj_das_100Hz_153.csv
已处理文件: /home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/xfj_das_100Hz_154.csv
已处理文件: /home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/xfj_das_100Hz_155.csv
已处理文件: /home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/xfj_das_100Hz_156.csv
已处理文件: /home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/xfj_das_100Hz_157.csv
已处理文件: /home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/xfj_das_100Hz_158.csv
已处理文件: /home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/xfj_das_100Hz_159.csv

# 对应三分量数据

In [15]:
import numpy as np
import h5py
import os
import glob
import argparse
import yaml
from tqdm import tqdm

def convert_npy_format(input_folder, output_file):
    """
    将多个CSV文件从原始格式转换为目标格式并合并
    
    参数:
        input_folder: 输入CSV文件所在的文件夹路径
        output_file: 输出CSV文件的路径
    """
    # 获取所有CSV文件
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    
    # 初始化一个空的数据框来存储所有数据
    all_data = pd.DataFrame()
    
    # 处理每个CSV文件
    with h5py.File(output_file, 'w') as h5f:
        for file_path in csv_files:
            
            # 读取CSV文件
            df = pd.read_csv(file_path, sep=',',header=None, names=['ch', 'phase_index', 'target', 'mag', 'phase'])
            S_df = df[df['phase']=='S'].copy()
            S_df.rename(columns={'target':'s_target'},inplace = True)

            P_df = df[df['phase']=='P'].copy()
            P_df.rename(columns={'target':'p_target'},inplace = True)
            merge_df = pd.merge(S_df, P_df, how='inner' , on=['ch'])


            npy_file_path = "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/data/"+file_path.split("/")[-1][:-4]+".npy"
            
            # 加载NPY文件
            try:
                data = np.load(npy_file_path)
            except FileNotFoundError:
                 continue
            # 检查数据维度
            if len(data.shape) != 2:
                print(f"跳过文件 {file_path}: 数据不是二维数组 (形状: {data.shape})")
                continue
            
            # 获取文件名（不含扩展名）作为数据集名称
            file_name = os.path.splitext(os.path.basename(file_path))[0]
            
            # 选择特定列
            columns =list(set(merge_df['ch'])) 
            for ch in columns:
                selected_data = np.array([data[ch],data[ch],data[ch]]).T
                # print(selected_data.shape)
                dataset_name = file_name + '_' + str(ch)
                print(dataset_name)
                # 将数据保存到H5文件
                h5f.create_dataset(dataset_name, data=selected_data)
                
            
        # 添加全局属性
    
    print(f"数据已成功保存到 {output_file}")
convert_npy_format( "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks" , "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_6w.h5")

xfj_das_100Hz_149_290
xfj_das_100Hz_149_291
xfj_das_100Hz_149_292
xfj_das_100Hz_149_293
xfj_das_100Hz_149_294
xfj_das_100Hz_149_295
xfj_das_100Hz_149_296
xfj_das_100Hz_149_297
xfj_das_100Hz_149_298
xfj_das_100Hz_149_299
xfj_das_100Hz_149_300
xfj_das_100Hz_149_301
xfj_das_100Hz_149_302
xfj_das_100Hz_149_303
xfj_das_100Hz_149_304
xfj_das_100Hz_149_305
xfj_das_100Hz_149_306
xfj_das_100Hz_149_307
xfj_das_100Hz_149_308
xfj_das_100Hz_149_309
xfj_das_100Hz_149_310
xfj_das_100Hz_149_311
xfj_das_100Hz_149_312
xfj_das_100Hz_149_313
xfj_das_100Hz_149_314
xfj_das_100Hz_149_315
xfj_das_100Hz_149_316
xfj_das_100Hz_149_317
xfj_das_100Hz_149_318
xfj_das_100Hz_149_319
xfj_das_100Hz_149_320
xfj_das_100Hz_149_321
xfj_das_100Hz_149_322
xfj_das_100Hz_149_323
xfj_das_100Hz_149_324
xfj_das_100Hz_149_325
xfj_das_100Hz_149_326
xfj_das_100Hz_149_327
xfj_das_100Hz_149_328
xfj_das_100Hz_149_329
xfj_das_100Hz_149_330
xfj_das_100Hz_149_331
xfj_das_100Hz_149_332
xfj_das_100Hz_149_333
xfj_das_100Hz_149_334
xfj_das_10

# 读取测试

In [18]:

import h5py
import numpy as np
with h5py.File("/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_6w.h5", 'r') as f:
    print(len(f.keys()))
    data = np.array(f.get("xfj_das_100Hz_114_1679"))
    print(data)
# data = np.load("/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/data/xfj_das_100Hz_114.npy")
# print(data)

31398
[[ 297.96252    297.96252    297.96252  ]
 [-291.8162    -291.8162    -291.8162   ]
 [  11.614355    11.614355    11.614355 ]
 ...
 [  -6.9804373   -6.9804373   -6.9804373]
 [ -37.445168   -37.445168   -37.445168 ]
 [ -23.184357   -23.184357   -23.184357 ]]


# 添加冗余部分噪声

In [19]:
import numpy as np
import h5py
import os
import glob
import argparse
import yaml
from tqdm import tqdm
import csv
import pandas as pd

def convert_npy_format(input_folder, output_file):
    """
    将多个CSV文件从原始格式转换为目标格式并合并
    
    参数:
        input_folder: 输入CSV文件所在的文件夹路径
        output_file: 输出CSV文件的路径
    """
    # 获取所有CSV文件
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    
    # 初始化一个空的数据框来存储所有数据
    all_data = pd.DataFrame()
    
    # 处理每个CSV文件
    with h5py.File(output_file, 'a') as h5f:
        for file_path in csv_files[::4]:
            
            # 读取CSV文件
            df = pd.read_csv(file_path, sep=',',header=None, names=['ch', 'phase_index', 'target', 'mag', 'phase'])
            S_df = df[df['phase']=='S']
            S_df.rename(columns={'target':'s_target'},inplace = True)

            P_df = df[df['phase']=='P']
            P_df.rename(columns={'target':'p_target'},inplace = True)
            merge_df = pd.merge(S_df, P_df, how='inner' , on=['ch'])


            npy_file_path = "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/data/"+file_path.split("/")[-1][:-4]+".npy"
            
            # 加载NPY文件
            # try:
            np_data = np.load(npy_file_path)
            # 获取文件名（不含扩展名）作为数据集名称
            file_name = os.path.splitext(os.path.basename(file_path))[0]
            
            # 选择特定列
            columns =[i for i in range(0,280)] + [i for i in range(2450 , 2800)]
            print(columns)
            for ch in columns:
                print(ch)
                selected_data = np.array([np_data[ch],np_data[ch],np_data[ch]]).T
                print(selected_data.shape)
                dataset_name = file_name + '_' + str(ch)
                print(dataset_name)
                # 将数据保存到H5文件
                try:

                    h5f.create_dataset(dataset_name, data=selected_data )
                    with open('/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_6w.csv', 'a+', newline='', encoding='utf-8') as file:
                        writer = csv.writer(file)
                        row_data = dataset_name,  np.nan , np.nan , np.nan,-0.2,"xfj3km", np.nan,"train",100
                        writer.writerow(row_data)
                except ValueError:
                    print("no ch")
                

            # except Exception as e:
            #     print(f"writing {dataset_name} 时出错: {str(e)}")
            
        # 添加全局属性
    
    print(f"数据已成功保存到 {output_file}")
convert_npy_format( "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks" , "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_6w.h5")

/tmp/ipykernel_58520/2887648945.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  S_df.rename(columns={'target':'s_target'},inplace = True)
/tmp/ipykernel_58520/2887648945.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'target':'p_target'},inplace = True)


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

/tmp/ipykernel_58520/2887648945.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  S_df.rename(columns={'target':'s_target'},inplace = True)
/tmp/ipykernel_58520/2887648945.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'target':'p_target'},inplace = True)


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

/tmp/ipykernel_58520/2887648945.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'target':'p_target'},inplace = True)


...

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

/tmp/ipykernel_58520/2887648945.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'target':'p_target'},inplace = True)


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

/tmp/ipykernel_58520/2887648945.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'target':'p_target'},inplace = True)


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

# 添加脚步噪声

In [20]:
import numpy as np
import h5py
import os
import glob
import argparse
import yaml
from tqdm import tqdm
import csv
import pandas as pd

def convert_npy_format(input_folder, output_file):
    """
    将多个CSV文件从原始格式转换为目标格式并合并
    
    参数:
        input_folder: 输入CSV文件所在的文件夹路径
        output_file: 输出CSV文件的路径
    """
    # 获取所有CSV文件
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    
    # 初始化一个空的数据框来存储所有数据
    all_data = pd.DataFrame()
    
    # 处理每个CSV文件
    with h5py.File(output_file, 'a') as h5f:
        for iev in range(85,86):
        
            npy_file_path = f"/home/disk/disk02/wzm/DAS_DL_Dataset/data/xfj/das_event_reorganize/xfj_das_re_eq_{iev}.npy"
            
            # 加载NPY文件
            # try:
            np_data = np.load(npy_file_path)
            # 获取文件名（不含扩展名）作为数据集名称
            file_name = f"xfj_das_100Hz_{iev}"
            
            # 选择特定列
            columns =[i for i in range(1000,1400)]
            print(columns)
            for ch in columns:
                print(ch)
                selected_data = np.array([np_data[ch],np_data[ch],np_data[ch]]).T
                print(selected_data.shape)
                dataset_name = file_name + '_' + str(ch)
                print(dataset_name)
                # 将数据保存到H5文件
                try:

                    h5f.create_dataset(dataset_name, data=selected_data )
                    with open('/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_6w.csv', 'a+', newline='', encoding='utf-8') as file:
                        writer = csv.writer(file)
                        row_data = dataset_name,  np.nan , np.nan , np.nan,-0.2,"xfj3km", np.nan,"train",100
                        writer.writerow(row_data)
                except ValueError:
                    print("no ch")
                

            # except Exception as e:
            #     print(f"writing {dataset_name} 时出错: {str(e)}")
            
        # 添加全局属性
    
    print(f"数据已成功保存到 {output_file}")
convert_npy_format( "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks" , "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_6w.h5")

[1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035, 1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047, 1048, 1049, 1050, 1051, 1052, 1053, 1054, 1055, 1056, 1057, 1058, 1059, 1060, 1061, 1062, 1063, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1078, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1096, 1097, 1098, 1099, 1100, 1101, 1102, 1103, 1104, 1105, 1106, 1107, 1108, 1109, 1110, 1111, 1112, 1113, 1114, 1115, 1116, 1117, 1118, 1119, 1120, 1121, 1122, 1123, 1124, 1125, 1126, 1127, 1128, 1129, 1130, 1131, 1132, 1133, 1134, 1135, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1143, 1144, 1145, 1146, 1147, 1148, 1149, 1150, 1151, 1152, 1153, 1154, 1155, 1156, 1157, 1158, 1159, 1160, 1161, 1162, 1163, 1164, 1165, 116

# 文件中添加所有数据

In [15]:
import numpy as np
import h5py
import os
import glob
import argparse
import yaml
from tqdm import tqdm
import csv
import pandas as pd

def convert_npy_format(input_folder, output_file):
    """
    将多个CSV文件从原始格式转换为目标格式并合并
    
    参数:
        input_folder: 输入CSV文件所在的文件夹路径
        output_file: 输出CSV文件的路径
    """
    # 获取所有CSV文件
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    
    # 初始化一个空的数据框来存储所有数据
    all_data = pd.DataFrame()
    
    # 处理每个CSV文件
    with h5py.File(output_file, 'a') as h5f:
        for iev in range(66,408):
        
            npy_file_path = f"/home/disk/disk02/wzm/DAS_DL_Dataset/data/xfj/das_event_reorganize/xfj_das_re_eq_{iev}.npy"
            
            # 加载NPY文件
            # try:
            np_data = np.load(npy_file_path)
            print(npy_file_path, np_data.shape)
            # 获取文件名（不含扩展名）作为数据集名称
            file_name = f"xfj_das_100Hz_{iev}"
            
            # 选择特定列
            columns =[i for i in range(0,3000)]
            # print(columns)
            for ch in columns:
                
                # 将数据保存到H5文件
                try:
                    # print(ch)
                    selected_data = np.array([np_data[ch],np_data[ch],np_data[ch]]).T
                    # print(selected_data.shape)
                    dataset_name = file_name + '_' + str(ch)
                    # print(dataset_name)
                    h5f.create_dataset(dataset_name, data=selected_data )
                    # with open('/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_6w.csv', 'a+', newline='', encoding='utf-8') as file:
                    #     writer = csv.writer(file)
                    #     row_data = dataset_name,  np.nan , np.nan , np.nan,-0.2,"xfj3km", np.nan,"train",100
                    #     writer.writerow(row_data)
                except ValueError:
                    print("no ch")
                

            # except Exception as e:
            #     print(f"writing {dataset_name} 时出错: {str(e)}")
            
        # 添加全局属性
    
    print(f"数据已成功保存到 {output_file}")
convert_npy_format( "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks" , "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_6w.h5")

/home/disk/disk02/wzm/DAS_DL_Dataset/data/xfj/das_event_reorganize/xfj_das_re_eq_66.npy (2944, 24000)
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch
no ch


KeyboardInterrupt: 